<div dir="rtl" style="text-align:right">

# תרגול 14 — מ־Seq2Seq ל־Transformer


```text
Encoder–Decoder
→ Teacher Forcing
→ Cross-Attention
→ Query, Key, Value
→ Scaled Dot-Product Attention
→ Causal Mask
→ Positional Encoding
→ Multi-Head Attention
→ Transformer Encoder
→ BERT-style Masking
```


</div>

<div dir="rtl" style="text-align:right">

## 0. Setup

</div>

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

print("PyTorch version:", torch.__version__)

In [ ]:
def plot_matrix(
    matrix,
    x_labels,
    y_labels,
    title,
    value_format=".2f",
    colorbar=True
):
    if torch.is_tensor(matrix):
        matrix = matrix.detach().cpu().numpy()
    else:
        matrix = np.asarray(matrix)

    figure_width = max(6, 0.9 * len(x_labels))
    figure_height = max(5, 0.75 * len(y_labels))

    plt.figure(figsize=(figure_width, figure_height))
    plt.imshow(matrix)

    plt.xticks(
        range(len(x_labels)),
        x_labels,
        rotation=45,
        ha="right"
    )
    plt.yticks(
        range(len(y_labels)),
        y_labels
    )

    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]

            if np.isfinite(value):
                label = format(value, value_format)
            else:
                label = "-∞"

            plt.text(
                col,
                row,
                label,
                ha="center",
                va="center"
            )

    plt.title(title)

    if colorbar:
        plt.colorbar()

    plt.tight_layout()
    plt.show()


def draw_encoder_decoder():
    plt.figure(figsize=(11, 4))
    plt.axis("off")

    blocks = [
        (0.10, "Input tokens\n[B, T_in]"),
        (0.30, "Embedding\n[B, T_in, E]"),
        (0.50, "Encoder\nstates"),
        (0.70, "Decoder\nstates"),
        (0.90, "Output logits\n[B, T_out, V]")
    ]

    for x_position, label in blocks:
        plt.text(
            x_position,
            0.55,
            label,
            ha="center",
            va="center",
            bbox={"boxstyle": "round,pad=0.5"}
        )

    for current, following in zip(
        blocks[:-1],
        blocks[1:]
    ):
        plt.annotate(
            "",
            xy=(following[0] - 0.08, 0.55),
            xytext=(current[0] + 0.08, 0.55),
            arrowprops={"arrowstyle": "->"}
        )

    plt.text(
        0.50,
        0.18,
        "Encoder represents the input sequence",
        ha="center"
    )
    plt.text(
        0.78,
        0.18,
        "Decoder generates one token at a time",
        ha="center"
    )

    plt.title("Encoder–Decoder Architecture")
    plt.show()


def draw_transformer_encoder():
    plt.figure(figsize=(6, 8))
    plt.axis("off")

    blocks = [
        (0.88, "Input Embeddings\n+ Positional Encoding"),
        (0.68, "Multi-Head\nSelf-Attention"),
        (0.52, "Residual Add\n+ LayerNorm"),
        (0.34, "Feed-Forward\nNetwork"),
        (0.18, "Residual Add\n+ LayerNorm"),
        (0.04, "Contextual Token\nRepresentations")
    ]

    for y_position, label in blocks:
        plt.text(
            0.5,
            y_position,
            label,
            ha="center",
            va="center",
            bbox={"boxstyle": "round,pad=0.5"}
        )

    for current, following in zip(
        blocks[:-1],
        blocks[1:]
    ):
        plt.annotate(
            "",
            xy=(0.5, following[0] + 0.055),
            xytext=(0.5, current[0] - 0.055),
            arrowprops={"arrowstyle": "->"}
        )

    plt.title("Transformer Encoder Block")
    plt.show()

<div dir="rtl" style="text-align:right">

# חלק א׳ — Encoder–Decoder ו־Teacher Forcing

נשתמש במשימה קטנה:

```text
Input:  A B C D
Target: D C B A
```

זו משימת Sequence-to-Sequence: רצף נכנס ורצף יוצא.

</div>

In [ ]:
draw_encoder_decoder()

<div dir="rtl" style="text-align:right">

## תרגיל 1 — Teacher Forcing

השלימו את `decoder_input_tokens` כך שה־target יוזז צעד אחד ימינה ויתחיל ב־`<BOS>`.

לאחר מכן צרו טבלה עם:

- step
- decoder input
- expected output

</div>

In [ ]:
input_tokens = ["A", "B", "C", "D"]

target_tokens = [
    "D",
    "C",
    "B",
    "A",
    "<EOS>"
]

# TODO:
# decoder_input_tokens = ...

# TODO:
# teacher_forcing_table = ...
# teacher_forcing_table

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
input_tokens = ["A", "B", "C", "D"]

target_tokens = [
    "D",
    "C",
    "B",
    "A",
    "<EOS>"
]

decoder_input_tokens = (
    ["<BOS>"]
    + target_tokens[:-1]
)

teacher_forcing_table = pd.DataFrame({
    "step": range(
        1,
        len(target_tokens) + 1
    ),
    "decoder_input": decoder_input_tokens,
    "expected_output": target_tokens
})

teacher_forcing_table
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 2 — Ideal Cross-Attention

במשימת היפוך, בנו מטריצת Attention אידאלית בגודל 4×4.

השורות הן output tokens והעמודות הן input tokens.

</div>

In [ ]:
# TODO:
# ideal_cross_attention = ...
# plot_matrix(...)

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
ideal_cross_attention = torch.eye(
    4
).flip(dims=[1])

plot_matrix(
    ideal_cross_attention,
    x_labels=input_tokens,
    y_labels=target_tokens[:-1],
    title="Ideal Cross-Attention for Sequence Reversal",
    value_format=".0f"
)
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ב׳ — Query, Key ו־Value

נשתמש במשפט:

```text
the cat sat on the mat
```

ה־embeddings הוגדרו ידנית כדי שאפשר יהיה לפרש את התוצאה.

</div>

In [ ]:
tokens = [
    "the",
    "cat",
    "sat",
    "on",
    "the",
    "mat"
]

# Toy embeddings chosen for visual interpretation.
# These vectors are not trained embeddings.
embeddings = torch.tensor(
    [
        [1.0, 0.0, 0.0, 0.0],  # the
        [0.0, 1.0, 0.0, 0.8],  # cat
        [0.0, 0.0, 1.0, 0.0],  # sat
        [0.0, 0.0, 0.5, 1.0],  # on
        [1.0, 0.0, 0.0, 0.0],  # the
        [0.0, 1.0, 0.0, 0.7]   # mat
    ],
    dtype=torch.float32
)

print("Token count:", len(tokens))
print("Embedding matrix shape:", embeddings.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 3 — Embedding Matrix

הציגו את מטריצת ה־embeddings כ־Heatmap.

</div>

In [ ]:
# TODO:
# plot_matrix(
#     embeddings,
#     ...
# )

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plot_matrix(
    embeddings,
    x_labels=[
        "dim 1",
        "dim 2",
        "dim 3",
        "dim 4"
    ],
    y_labels=tokens,
    title="Toy Token Embeddings"
)
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ג׳ — Scaled Dot-Product Attention

לצורך ההדגמה נשתמש ב:

```text
Q = K = V = embeddings
```

במודל מאומן, Q, K ו־V נוצרים באמצעות שכבות Linear נלמדות.

</div>

In [ ]:
# For a transparent demonstration, use Q = K = V = embeddings.
# In a trained Transformer, Q, K and V are learned linear projections.
Q = embeddings
K = embeddings
V = embeddings

d_k = Q.shape[-1]

raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(d_k)

attention_weights = torch.softmax(
    scaled_scores,
    dim=-1
)

context_vectors = attention_weights @ V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
print("Score matrix shape:", scaled_scores.shape)
print("Attention matrix shape:", attention_weights.shape)
print("Context matrix shape:", context_vectors.shape)
print("Row sums:", attention_weights.sum(dim=-1))

<div dir="rtl" style="text-align:right">

## תרגיל 4 — ויזואליזציה של Attention

הציגו:

1. Raw `QKᵀ` scores.
2. Scaled scores.
3. Attention weights.
4. בדקו שכל שורה ב־Attention מסתכמת ל־1.

</div>

In [ ]:
# TODO:
# plot raw_scores
# plot scaled_scores
# plot attention_weights
# print row sums

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plot_matrix(
    raw_scores,
    x_labels=tokens,
    y_labels=tokens,
    title="Raw QKᵀ Scores"
)

plot_matrix(
    scaled_scores,
    x_labels=tokens,
    y_labels=tokens,
    title="Scaled Scores: QKᵀ / √dₖ"
)

plot_matrix(
    attention_weights,
    x_labels=tokens,
    y_labels=tokens,
    title="Self-Attention Weights"
)

print(
    "Attention row sums:",
    attention_weights.sum(dim=-1)
)
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 5 — למי `cat` מתייחס?

צרו טבלה שמציגה את משקלי ה־Attention של Query בשם `cat` מול כל Key.

מיינו מהמשקל הגבוה לנמוך.

</div>

In [ ]:
# TODO:
# query_index = ...
# cat_attention = ...
# cat_attention

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
query_token = "cat"
query_index = tokens.index(query_token)

cat_attention = pd.DataFrame({
    "key_token": tokens,
    "attention_weight": (
        attention_weights[query_index]
        .detach()
        .numpy()
    )
}).sort_values(
    "attention_weight",
    ascending=False
)

cat_attention
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ד׳ — Causal Mask

Decoder אוטורגרסיבי אינו רשאי לראות tokens עתידיים.

</div>

In [ ]:
sequence_length = len(tokens)

future_positions = torch.triu(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool
    ),
    diagonal=1
)

allowed_positions = (~future_positions).float()

masked_scores = scaled_scores.masked_fill(
    future_positions,
    float("-inf")
)

causal_attention_weights = torch.softmax(
    masked_scores,
    dim=-1
)

print("Causal attention row sums:")
print(causal_attention_weights.sum(dim=-1))

<div dir="rtl" style="text-align:right">

## תרגיל 6 — מסכה ומשקלי Attention

הציגו:

1. Visibility Matrix שבה 1=מותר ו־0=אסור.
2. Attention Weights אחרי Causal Mask.
3. ודאו שכל הערכים מעל האלכסון הם 0 לאחר Softmax.

</div>

In [ ]:
# TODO:
# plot allowed_positions
# plot causal_attention_weights

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plot_matrix(
    allowed_positions,
    x_labels=tokens,
    y_labels=tokens,
    title="Causal Mask: 1 = Visible, 0 = Hidden",
    value_format=".0f"
)

plot_matrix(
    causal_attention_weights,
    x_labels=tokens,
    y_labels=tokens,
    title="Self-Attention after Causal Mask"
)

print(
    "Upper triangle after Softmax:",
    causal_attention_weights[
        future_positions
    ]
)
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 7 — `sat` לפני ואחרי Mask

השוו בטבלה את Attention של Query בשם `sat` לפני ואחרי Causal Mask.

</div>

In [ ]:
# TODO:
# token_index = ...
# comparison = ...
# comparison

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
token_index = tokens.index("sat")

comparison = pd.DataFrame({
    "token": tokens,
    "without_mask": (
        attention_weights[token_index]
        .detach()
        .numpy()
    ),
    "with_causal_mask": (
        causal_attention_weights[token_index]
        .detach()
        .numpy()
    )
})

comparison
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ה׳ — Positional Encoding

Self-Attention לבדו אינו יודע את סדר המילים.

נוסיף Sinusoidal Positional Encoding ל־embeddings.

</div>

In [ ]:
def sinusoidal_positional_encoding(
    sequence_length,
    d_model
):
    positions = torch.arange(
        sequence_length,
        dtype=torch.float32
    ).unsqueeze(1)

    even_dimensions = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    frequency_scale = torch.exp(
        even_dimensions
        * (
            -math.log(10000.0)
            / d_model
        )
    )

    positional_encoding = torch.zeros(
        sequence_length,
        d_model
    )

    positional_encoding[:, 0::2] = torch.sin(
        positions * frequency_scale
    )

    positional_encoding[:, 1::2] = torch.cos(
        positions * frequency_scale
    )

    return positional_encoding

<div dir="rtl" style="text-align:right">

## תרגיל 8 — Positional Encoding

1. צרו Positional Encoding באותו shape כמו embeddings.
2. הציגו אותו כ־Heatmap.
3. ציירו כל ממד לאורך הרצף בגרף נפרד.
4. הוסיפו אותו ל־embeddings.

</div>

In [ ]:
# TODO:
# positional_encoding = ...
# plot heatmap
# plot each dimension
# embeddings_with_position = ...

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
positional_encoding = sinusoidal_positional_encoding(
    sequence_length=len(tokens),
    d_model=embeddings.shape[1]
)

plot_matrix(
    positional_encoding,
    x_labels=[
        "dim 1",
        "dim 2",
        "dim 3",
        "dim 4"
    ],
    y_labels=[
        f"position {index}"
        for index in range(len(tokens))
    ],
    title="Sinusoidal Positional Encoding"
)

for dimension in range(
    positional_encoding.shape[1]
):
    plt.figure(figsize=(7, 4))
    plt.plot(
        range(len(tokens)),
        positional_encoding[:, dimension],
        marker="o"
    )
    plt.xlabel("Position")
    plt.ylabel("Encoding value")
    plt.title(
        f"Positional Encoding — Dimension {dimension}"
    )
    plt.show()

embeddings_with_position = (
    embeddings
    + positional_encoding
)

print(
    "Embeddings with position shape:",
    embeddings_with_position.shape
)
```

</details>

<div dir="rtl" style="text-align:right">

## תרגיל 9 — שתי הופעות של `the`

השוו Cosine Similarity בין שני מופעי `the`:

1. לפני Positional Encoding.
2. אחרי Positional Encoding.

</div>

In [ ]:
# TODO:
# raw_similarity = ...
# position_aware_similarity = ...
# print(...)

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
raw_similarity = F.cosine_similarity(
    embeddings[0].unsqueeze(0),
    embeddings[4].unsqueeze(0)
).item()

position_aware_similarity = (
    F.cosine_similarity(
        embeddings_with_position[0].unsqueeze(0),
        embeddings_with_position[4].unsqueeze(0)
    ).item()
)

print(
    "Similarity before positional encoding:",
    raw_similarity
)

print(
    "Similarity after positional encoding:",
    position_aware_similarity
)
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ו׳ — Multi-Head Attention

נשתמש בשני heads ונציג Heatmap נפרדת לכל head.

ה־heads עדיין לא אומנו, ולכן מטרת התרגיל היא להבין shapes ומבנה.

</div>

In [ ]:
d_model = embeddings_with_position.shape[1]

multihead_attention = nn.MultiheadAttention(
    embed_dim=d_model,
    num_heads=2,
    dropout=0.0,
    batch_first=True
)

input_batch = embeddings_with_position.unsqueeze(0)

attention_output, per_head_weights = multihead_attention(
    input_batch,
    input_batch,
    input_batch,
    need_weights=True,
    average_attn_weights=False
)

print("Input shape:", input_batch.shape)
print("Output shape:", attention_output.shape)
print("Per-head weights shape:", per_head_weights.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 10 — Attention Head לכל גרף

הציגו:

- Head 1.
- Head 2.

</div>

In [ ]:
# TODO:
# plot per_head_weights[0, 0]
# plot per_head_weights[0, 1]

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plot_matrix(
    per_head_weights[0, 0],
    x_labels=tokens,
    y_labels=tokens,
    title="Multi-Head Attention — Head 1"
)

plot_matrix(
    per_head_weights[0, 1],
    x_labels=tokens,
    y_labels=tokens,
    title="Multi-Head Attention — Head 2"
)
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ז׳ — Transformer Encoder Block

</div>

In [ ]:
draw_transformer_encoder()

In [ ]:
transformer_encoder_layer = nn.TransformerEncoderLayer(
    d_model=4,
    nhead=2,
    dim_feedforward=16,
    dropout=0.0,
    batch_first=True
)

transformer_encoder_layer.eval()

with torch.no_grad():
    encoded_tokens = transformer_encoder_layer(
        input_batch
    )

print("Input shape:", input_batch.shape)
print("Encoded output shape:", encoded_tokens.shape)

<div dir="rtl" style="text-align:right">

## תרגיל 11 — Shapes לפני ואחרי Transformer

ענו:

1. מה Shape הקלט?
2. מה Shape הפלט?
3. האם ה־Shape נשמר?
4. האם ערכי הווקטורים נשארים זהים?

</div>

In [ ]:
# TODO:
# השוו input_batch ו-encoded_tokens
# חשבו mean absolute difference

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
print("Input shape:", input_batch.shape)
print("Output shape:", encoded_tokens.shape)

mean_absolute_difference = (
    encoded_tokens - input_batch
).abs().mean().item()

print(
    "Mean absolute difference:",
    mean_absolute_difference
)
```

</details>

<div dir="rtl" style="text-align:right">

# חלק ח׳ — BERT-style Masking

BERT הוא Encoder דו־כיווני.

- אין Causal Mask.
- `[MASK]` גלוי למודל.
- `[PAD]` מוסתר באמצעות Padding Mask.

</div>

In [ ]:
bert_tokens = [
    "[CLS]",
    "the",
    "cat",
    "sat",
    "on",
    "the",
    "[MASK]",
    "[SEP]",
    "[PAD]",
    "[PAD]"
]

padding_mask = torch.tensor(
    [
        token == "[PAD]"
        for token in bert_tokens
    ],
    dtype=torch.bool
)

visible_keys = (~padding_mask).float()

bert_key_visibility = visible_keys.repeat(
    len(bert_tokens),
    1
)

decoder_visibility = torch.tril(
    torch.ones(
        len(bert_tokens),
        len(bert_tokens)
    )
)

print("BERT tokens:", bert_tokens)
print("Padding mask:", padding_mask)

<div dir="rtl" style="text-align:right">

## תרגיל 12 — BERT מול Decoder

הציגו:

1. BERT Key Visibility.
2. Decoder Causal Visibility.
3. הסבירו למה `[MASK]` גלוי ולמה `[PAD]` מוסתר.

</div>

In [ ]:
# TODO:
# plot bert_key_visibility
# plot decoder_visibility

<details dir="rtl">
<summary>פתרון מוסתר</summary>

```python
plot_matrix(
    bert_key_visibility,
    x_labels=bert_tokens,
    y_labels=bert_tokens,
    title="BERT Padding Mask: 1 = Key Visible",
    value_format=".0f"
)

plot_matrix(
    decoder_visibility,
    x_labels=bert_tokens,
    y_labels=bert_tokens,
    title="Decoder Causal Visibility",
    value_format=".0f"
)
```

</details>